# 08b — EES Capital Baseline for Mountain West Census Tracts

Builds Environmental (E), Economic (Ec), and Social (S) capital scores for
census tracts in the 7 target Level-III ecoregions of the Mountain West.

**Target ecoregions (US_L3CODE):** 17 Middle Rockies, 18 Wyoming Basin,
20 Colorado Plateaus, 21 Southern Rockies, 25 High Plains,
43 Northwestern Great Plains, 80 Northern Basin and Range.

**Outputs:**
- `data/processed/mw_tract_ees_scores.parquet` — tract-level indicators + scores
- `data/processed/mw_ecoregion_ees_summary.csv` — ecoregion-level summary
- `data/processed/figures/ees_baseline_map.html` — Folium choropleth
- `data/processed/network_metadata.json` — updated with `ees_baseline` block


In [1]:
import io, json, time, warnings, zipfile
from pathlib import Path

import folium
import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
import os

warnings.filterwarnings('ignore')

ROOT = Path('/Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map')

# ── API keys ──────────────────────────────────────────────────────────────────
load_dotenv(ROOT / '.env')
CENSUS_KEY = os.environ.get('CENSUS_API_KEY', '')
assert CENSUS_KEY, 'CENSUS_API_KEY not found in .env'
print(f'CENSUS_API_KEY loaded: {CENSUS_KEY[:8]}...')
print()

# ── Source files ──────────────────────────────────────────────────────────────
eco_mw     = gpd.read_file(ROOT / 'data/processed/mw_ecoregions.geojson')
xwalk      = gpd.read_file(ROOT / 'data/processed/ecoregion_ba_crosswalk.geojson')
buses_gdf  = gpd.read_file(ROOT / 'data/processed/synthetic_buses.geojson')
plants_gdf = gpd.read_file(ROOT / 'data/processed/power_plants_with_ba.geojson')
with open(ROOT / 'data/processed/network_metadata.json') as fh:
    meta = json.load(fh)

print('=== mw_ecoregions.geojson ===')
print(f'  Shape: {eco_mw.shape}  CRS: {eco_mw.crs}')
print(f'  Columns: {list(eco_mw.columns)}')
print(f'  Unique US_L3CODE: {sorted(eco_mw["US_L3CODE"].unique())}')
print()
print('=== ecoregion_ba_crosswalk.geojson ===')
print(f'  Shape: {xwalk.shape}  CRS: {xwalk.crs}')
print(f'  Columns: {list(xwalk.columns)}')
print()
print('=== synthetic_buses.geojson ===')
print(f'  Shape: {buses_gdf.shape}  CRS: {buses_gdf.crs}')
print(f'  Columns: {list(buses_gdf.columns)}')
print()
print('=== power_plants_with_ba.geojson ===')
print(f'  Shape: {plants_gdf.shape}  CRS: {plants_gdf.crs}')
print(f'  Capacity range: {plants_gdf["capacity_mw"].min():.1f} – {plants_gdf["capacity_mw"].max():.1f} MW')
print()
print('=== network_metadata.json ===')
print(f'  Top-level keys: {list(meta.keys())}')


CENSUS_API_KEY loaded: ffba151d...



=== mw_ecoregions.geojson ===
  Shape: (128, 8)  CRS: EPSG:4326
  Columns: ['US_L3CODE', 'US_L3NAME', 'NA_L3CODE', 'NA_L3NAME', 'NA_L2CODE', 'NA_L2NAME', 'STATE_NAME', 'geometry']
  Unique US_L3CODE: ['12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '25', '26', '27', '41', '42', '43', '44', '46', '80']

=== ecoregion_ba_crosswalk.geojson ===
  Shape: (183, 6)  CRS: EPSG:4326
  Columns: ['ecoregion_code', 'ecoregion_name', 'ba_code', 'ba_name', 'area_km2', 'geometry']

=== synthetic_buses.geojson ===
  Shape: (500, 10)  CRS: EPSG:4326
  Columns: ['bus_id', 'ba_code', 'lon', 'lat', 'n_counties', 'population', 'generation_cap_mw', 'load_mw', 'role', 'geometry']

=== power_plants_with_ba.geojson ===
  Shape: (15034, 29)  CRS: EPSG:4326
  Capacity range: 0.1 – 1499.4 MW

=== network_metadata.json ===
  Top-level keys: ['grid_size_m', 'data_vintage_year', 'pct_mw_retained_at_filter', 'atb_scenario', 'atb_version', 'n_zones', 'scenarios_completed', 'e4st_v2', 'giant_com

## Target Ecoregion Filter

Filter `mw_ecoregions.geojson` to the 7 target Level-III ecoregions.
Adjacent ecoregions clipped into the Mountain West bbox but not in scope
(e.g. Snake River Plain, Central Basin and Range) are excluded here.


In [2]:
TARGET_ECO_CODES = ['17', '18', '20', '21', '25', '43', '80']
TARGET_ECO_MAP = {
    '17': 'Middle Rockies',
    '18': 'Wyoming Basin',
    '20': 'Colorado Plateaus',
    '21': 'Southern Rockies',
    '25': 'High Plains',
    '43': 'Northwestern Great Plains',
    '80': 'Northern Basin and Range',
}

eco_target = eco_mw[eco_mw['US_L3CODE'].isin(TARGET_ECO_CODES)].copy()

print(f'Target ecoregions: {len(eco_target)} polygon fragments across '
      f'{eco_target["US_L3CODE"].nunique()} ecoregion types')
print()
for code in TARGET_ECO_CODES:
    sub = eco_target[eco_target['US_L3CODE'] == code]
    print(f'  {code}: {TARGET_ECO_MAP[code]}  ({len(sub)} fragment(s))')


Target ecoregions: 72 polygon fragments across 7 ecoregion types

  17: Middle Rockies  (22 fragment(s))
  18: Wyoming Basin  (8 fragment(s))
  20: Colorado Plateaus  (4 fragment(s))
  21: Southern Rockies  (8 fragment(s))
  25: High Plains  (13 fragment(s))
  43: Northwestern Great Plains  (9 fragment(s))
  80: Northern Basin and Range  (8 fragment(s))


## Step 1 — Census Tract Geometries

Download 2020 TIGER/Line tract shapefiles for 9 Mountain West states from the Census FTP.
Cache to `data/raw/census_tracts/mw_tracts_2020.parquet`.
Clip tracts to study area by assigning each tract's centroid to an ecoregion via
point-in-polygon; tracts whose centroid falls outside all 7 target ecoregions are
retained in `tracts_all` but excluded from `tracts_study`.

States: WY, CO, MT, UT, NM, ID, NV, NE, SD (FIPS 56, 08, 30, 49, 35, 16, 32, 31, 46).


In [3]:
import tempfile

MW_STATE_FIPS = {
    'WY': '56', 'CO': '08', 'MT': '30', 'UT': '49', 'NM': '35',
    'ID': '16', 'NV': '32', 'NE': '31', 'SD': '46',
}

TRACT_DIR  = ROOT / 'data/raw/census_tracts'
TRACT_DIR.mkdir(parents=True, exist_ok=True)
TRACT_FILE = TRACT_DIR / 'mw_tracts_2020.parquet'

if TRACT_FILE.exists():
    tracts_raw = gpd.read_parquet(TRACT_FILE)
    print(f'Loaded cached tracts: {len(tracts_raw):,} rows')
else:
    state_gdfs = []
    for abbr, fips in MW_STATE_FIPS.items():
        url = (
            'https://www2.census.gov/geo/tiger/TIGER2020/TRACT/'
            f'tl_2020_{fips}_tract.zip'
        )
        print(f'  Downloading {abbr} (FIPS {fips}) ...', end=' ', flush=True)
        ok = False
        for attempt in range(2):
            try:
                r = requests.get(url, timeout=120)
                r.raise_for_status()
                # Extract to temp dir so pyogrio can read from disk (not ZipExtFile)
                with tempfile.TemporaryDirectory() as tmpdir:
                    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                        z.extractall(tmpdir)
                    shp_path = next(Path(tmpdir).glob('*.shp'))
                    gdf = gpd.read_file(shp_path)
                gdf = gdf.to_crs('EPSG:4326')
                gdf['state_abbr'] = abbr
                gdf['state_fips'] = fips
                keep = ['GEOID', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'NAME',
                        'ALAND', 'INTPTLAT', 'INTPTLON', 'state_abbr', 'state_fips', 'geometry']
                state_gdfs.append(gdf[[c for c in keep if c in gdf.columns]])
                print(f'OK ({len(gdf):,} tracts)')
                ok = True
                break
            except Exception as e:
                if attempt == 0:
                    print('retry... ', end='', flush=True)
                    time.sleep(5)
                else:
                    print(f'FAILED: {e}')
        if not ok:
            print(f'  WARNING: {abbr} tracts missing — continuing')

    if not state_gdfs:
        raise RuntimeError('No tract data downloaded. Check network connectivity.')
    tracts_raw = gpd.GeoDataFrame(pd.concat(state_gdfs, ignore_index=True), crs='EPSG:4326')
    tracts_raw.to_parquet(TRACT_FILE)
    print(f'\n  Cached {len(tracts_raw):,} tracts → {TRACT_FILE.name}')

print(f'\nTotal tracts: {len(tracts_raw):,}  |  CRS: {tracts_raw.crs}')

# ── Assign ecoregion via centroid point-in-polygon ────────────────────────────
print('Assigning ecoregion codes via tract centroid ...')

eco_for_join = (
    eco_target[['US_L3CODE', 'US_L3NAME', 'geometry']]
    .rename(columns={'US_L3CODE': 'ecoregion_code', 'US_L3NAME': 'ecoregion_name'})
    .copy()
)

# Centroids in EPSG:5070 for accuracy, then back to 4326 for sjoin
centroids_4326 = gpd.GeoDataFrame(
    {'GEOID': tracts_raw['GEOID']},
    geometry=tracts_raw.to_crs('EPSG:5070').geometry.centroid.to_crs('EPSG:4326'),
    crs='EPSG:4326',
)

joined = gpd.sjoin(
    centroids_4326,
    eco_for_join,
    how='left',
    predicate='within',
).drop(columns=['index_right'])
joined = joined.drop_duplicates(subset=['GEOID'], keep='first')

tracts_all   = tracts_raw.merge(joined[['GEOID', 'ecoregion_code', 'ecoregion_name']], on='GEOID', how='left')
tracts_study = tracts_all[tracts_all['ecoregion_code'].notna()].copy().reset_index(drop=True)

# Pre-compute EPSG:5070 tract centroids (reused in Steps 3–5)
tract_cent_5070 = gpd.GeoDataFrame(
    {'GEOID': tracts_study['GEOID']},
    geometry=tracts_study.to_crs('EPSG:5070').geometry.centroid,
    crs='EPSG:5070',
)

print(f'\n  Total 9-state tracts:                {len(tracts_all):>6,}')
print(f'  Tracts within study-area ecoregions: {len(tracts_study):>6,}')
print()
eco_dist = (
    tracts_study.groupby(['ecoregion_code', 'ecoregion_name']).size()
    .reset_index(name='n_tracts').sort_values('ecoregion_code')
)
print(eco_dist.to_string(index=False))


OK (160 tracts)

OK (1,447 tracts)

OK (319 tracts)

OK (716 tracts)

OK (612 tracts)

OK (456 tracts)

OK (779 tracts)

OK (553 tracts)

OK (242 tracts)

  Cached 5,284 tracts → mw_tracts_2020.parquet

Total tracts: 5,284  |  CRS: EPSG:4326
Assigning ecoregion codes via tract centroid ...



  Total 9-state tracts:                 5,284
  Tracts within study-area ecoregions:  1,690

ecoregion_code            ecoregion_name  n_tracts
            17            Middle Rockies       157
            18             Wyoming Basin        74
            20         Colorado Plateaus       112
            21          Southern Rockies       204
            25               High Plains       949
            43 Northwestern Great Plains       175
            80  Northern Basin and Range        19


## Step 2 — ACS Economic and Social Indicators

Pull ACS 5-year 2022 data for all tracts in the 9 states.
- **B-table endpoint:** `https://api.census.gov/data/2022/acs/acs5`
- **S-table endpoint:** `https://api.census.gov/data/2022/acs/acs5/subject`

Census sentinel value −666666666 → NaN. Tracts with >3 missing primary indicators
are flagged and excluded from scoring.

**Note on government employment:** The user spec cited `C24050_003E` (government workers),
but C24050 encodes industry sectors (row 3 = Agriculture). Government class-of-worker is
in **C24060**: rows 5 (local) + 6 (state) + 7 (federal) / row 1 (total). C24060 is used here.


In [4]:
ACS_CACHE = ROOT / 'data/raw/acs_tract_2022.parquet'

B_VARS = {
    'B01003_001E': 'population',
    'B19013_001E': 'median_hh_income',
    'B25002_003E': 'vacant_units',
    'B25002_001E': 'total_units',
    'B23025_004E': 'employed',
    'B23025_003E': 'labor_force',
    'B25035_001E': 'median_year_built',
    'C24060_001E': 'total_workers',
    'C24060_005E': 'local_govt_workers',
    'C24060_006E': 'state_govt_workers',
    'C24060_007E': 'federal_govt_workers',
}

S_VARS = {
    'S1701_C03_001E': 'poverty_rate',
    'S1501_C02_015E': 'pct_bachelor_plus',
    'S2701_C03_001E': 'pct_health_insurance',
    'S2801_C02_014E': 'pct_internet',
}

B_BASE = 'https://api.census.gov/data/2022/acs/acs5'
S_BASE = 'https://api.census.gov/data/2022/acs/acs5/subject'
SENTINEL = -666666000

PRIMARY_INDICATORS = [
    'median_hh_income', 'poverty_rate', 'vacancy_rate', 'employment_rate',
    'pct_bachelor_plus', 'pct_health_insurance', 'median_year_built', 'pct_internet',
]


def fetch_acs(base_url, var_dict, fips, key, retries=2):
    var_str = ','.join(var_dict.keys())
    url = f'{base_url}?get={var_str}&for=tract:*&in=state:{fips}&key={key}'
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            data = r.json()
            df = pd.DataFrame(data[1:], columns=data[0])
            df['GEOID'] = df['state'] + df['county'] + df['tract']
            df = df.drop(columns=['state', 'county', 'tract'], errors='ignore')
            df = df.rename(columns=var_dict)
            for col in var_dict.values():
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    df.loc[df[col] <= SENTINEL, col] = np.nan
            return df
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(5)
            else:
                return None


if ACS_CACHE.exists():
    acs_df = pd.read_parquet(ACS_CACHE)
    print(f'Loaded cached ACS data: {acs_df.shape}')
else:
    all_b, all_s = [], []
    for abbr, fips in MW_STATE_FIPS.items():
        print(f'  {abbr} ...', end=' ', flush=True)
        b_df = fetch_acs(B_BASE, B_VARS, fips, CENSUS_KEY)
        if b_df is None:
            print('B-table FAILED ', end='')
            b_df = pd.DataFrame({'GEOID': pd.Series(dtype=str)})
        s_df = fetch_acs(S_BASE, S_VARS, fips, CENSUS_KEY)
        if s_df is None:
            print('S-table FAILED ', end='')
            s_df = pd.DataFrame({'GEOID': pd.Series(dtype=str)})
        all_b.append(b_df)
        all_s.append(s_df)
        print('OK')
        time.sleep(0.5)

    b_all = pd.concat(all_b, ignore_index=True)
    s_all = pd.concat(all_s, ignore_index=True)
    acs_df = b_all.merge(s_all, on='GEOID', how='outer')

    # Derived rates
    acs_df['vacancy_rate'] = (
        acs_df['vacant_units'] / acs_df['total_units'].replace(0, np.nan) * 100
    )
    acs_df['employment_rate'] = (
        acs_df['employed'] / acs_df['labor_force'].replace(0, np.nan) * 100
    )
    acs_df['govt_employment_share'] = (
        (acs_df['local_govt_workers'].fillna(0)
         + acs_df['state_govt_workers'].fillna(0)
         + acs_df['federal_govt_workers'].fillna(0))
        / acs_df['total_workers'].replace(0, np.nan) * 100
    )
    acs_df['n_missing'] = acs_df[PRIMARY_INDICATORS].isna().sum(axis=1).astype(int)

    acs_df.to_parquet(ACS_CACHE)
    print(f'\n  Cached ACS: {acs_df.shape} → {ACS_CACHE.name}')

# Merge onto study tracts
tracts_study = tracts_study.merge(acs_df, on='GEOID', how='left')
# Tracts with no ACS row: set n_missing = max (exclude from scoring)
n_ind = len(PRIMARY_INDICATORS)
if 'n_missing' not in tracts_study.columns:
    tracts_study['n_missing'] = n_ind
else:
    tracts_study['n_missing'] = tracts_study['n_missing'].fillna(n_ind).astype(int)

# Add derived rate columns if missing (when loading from cache)
if 'vacancy_rate' not in tracts_study.columns:
    tracts_study['vacancy_rate'] = (
        tracts_study['vacant_units'] / tracts_study['total_units'].replace(0, np.nan) * 100
    )
if 'employment_rate' not in tracts_study.columns:
    tracts_study['employment_rate'] = (
        tracts_study['employed'] / tracts_study['labor_force'].replace(0, np.nan) * 100
    )
if 'govt_employment_share' not in tracts_study.columns:
    tracts_study['govt_employment_share'] = (
        (tracts_study.get('local_govt_workers', pd.Series(0, index=tracts_study.index)).fillna(0)
         + tracts_study.get('state_govt_workers', pd.Series(0, index=tracts_study.index)).fillna(0)
         + tracts_study.get('federal_govt_workers', pd.Series(0, index=tracts_study.index)).fillna(0))
        / tracts_study['total_workers'].replace(0, np.nan) * 100
    )

n_usable  = (tracts_study['n_missing'] <= 3).sum()
n_flagged = (tracts_study['n_missing'] >  3).sum()
print(f'\nStudy-area tracts: {len(tracts_study):,}')
print(f'  Usable (≤3 missing): {n_usable:,}')
print(f'  Flagged (>3 missing): {n_flagged:,}')
print()
print('Missing-indicator distribution:')
print(tracts_study['n_missing'].value_counts().sort_index().to_string())


  WY ... 

OK


  CO ... 

OK


  MT ... 

...

  NE ... 

OK


  SD ... 

OK



  Cached ACS: (5284, 20) → acs_tract_2022.parquet

Study-area tracts: 1,690
  Usable (≤3 missing): 1,676
  Flagged (>3 missing): 14

Missing-indicator distribution:
n_missing
0    1654
1      21
2       1
5       1
7       1
8      12


## Step 3 — Energy Infrastructure Indicators

**(a) Total generation capacity (MW) within 50 km** of each tract centroid.
Source: `power_plants_with_ba.geojson` — all 15,034 EIA plants, status=OP.
All distance calculations in EPSG:5070 (Albers Equal Area CONUS).

**(b) Distance to nearest synthetic bus (km)** — transmission network access proxy.
Source: `synthetic_buses.geojson` (500 buses).


In [5]:
print('Computing energy infrastructure indicators ...')

# ── (a) Generator capacity within 50 km ──────────────────────────────────────
gen_5070 = plants_gdf[['plantid', 'capacity_mw', 'geometry']].to_crs('EPSG:5070')
gen_5070 = gen_5070.dropna(subset=['geometry', 'capacity_mw'])

# 50 km buffer around each tract centroid
buffers = tract_cent_5070.copy()
buffers['geometry'] = tract_cent_5070.geometry.buffer(50_000)

gen_in_buf = gpd.sjoin(
    gen_5070,
    buffers[['GEOID', 'geometry']],
    how='inner',
    predicate='within',
)
gen_cap_by_tract = (
    gen_in_buf.groupby('GEOID')['capacity_mw']
    .sum()
    .reset_index()
    .rename(columns={'capacity_mw': 'gen_cap_50km_mw'})
)

# ── (b) Distance to nearest synthetic bus ─────────────────────────────────────
buses_5070 = buses_gdf.to_crs('EPSG:5070')[['bus_id', 'geometry']]

nearest = gpd.sjoin_nearest(
    tract_cent_5070,
    buses_5070,
    how='left',
    distance_col='dist_to_bus_m',
)
nearest = nearest.drop_duplicates(subset=['GEOID'], keep='first')
nearest['dist_to_bus_km'] = nearest['dist_to_bus_m'] / 1000.0

# ── Merge into tracts_study ───────────────────────────────────────────────────
tracts_study = tracts_study.merge(gen_cap_by_tract, on='GEOID', how='left')
tracts_study['gen_cap_50km_mw'] = tracts_study['gen_cap_50km_mw'].fillna(0.0)
tracts_study = tracts_study.merge(nearest[['GEOID', 'dist_to_bus_km']], on='GEOID', how='left')

print(f'  Generator capacity within 50 km:')
print(f'    Median {tracts_study["gen_cap_50km_mw"].median():.0f} MW  |  '
      f'Max {tracts_study["gen_cap_50km_mw"].max():.0f} MW  |  '
      f'Zero-MW tracts: {(tracts_study["gen_cap_50km_mw"] == 0).sum():,}')
print(f'  Distance to nearest synthetic bus:')
print(f'    Median {tracts_study["dist_to_bus_km"].median():.1f} km  |  '
      f'Max {tracts_study["dist_to_bus_km"].max():.1f} km')


Computing energy infrastructure indicators ...


  Generator capacity within 50 km:
    Median 2039 MW  |  Max 5185 MW  |  Zero-MW tracts: 163
  Distance to nearest synthetic bus:
    Median 33.6 km  |  Max 212.9 km


## Step 4 — Environmental Indicators (Proxy Approach)

Three sub-indicators assembled from vector/tabular sources (no raster).
Each is flagged below with a **TODO** for replacement in future work.

### 4a — Ecoregion base score (structural prior)
Scale 0–10 from EPA Level III ecoregion descriptions (2011).
Higher = better ecological condition / lower development pressure.

### 4b — HUC-8 water stress (arid ecoregion fraction)
Fraction of each HUC-8 watershed area that overlaps with arid ecoregions
(Wyoming Basin 18, Colorado Plateaus 20, High Plains 25).
Higher fraction = higher water stress = lower environmental score.
Source: USGS Watershed Boundary Dataset REST API.

### 4c — Development pressure
Vacancy rate proxy: low vacancy in rural ecoregion signals growth pressure
on natural land. Higher vacancy = lower pressure = better environmental score.
Source: ACS B25002 (already fetched in Step 2).


In [6]:
# ── 4a: Ecoregion base environmental score ────────────────────────────────────
# Structural prior — EPA Level III Ecoregion Descriptions, 2011
# https://www.epa.gov/eco-research/ecoregions-north-america
ECO_BASE_SCORES = {
    '21': 9.0,   # Southern Rockies — highest % protected land, intact alpine/subalpine
    '17': 8.5,   # Middle Rockies — intact coniferous forest, low urban pressure
    '25': 6.0,   # High Plains — mixed shortgrass prairie + cropland, moderate fragmentation
    '43': 5.5,   # Northwestern Great Plains — mixed-grass prairie, cultivated cropland
    '80': 5.0,   # Northern Basin and Range — semi-arid shrubland, ranching pressure
    '18': 4.5,   # Wyoming Basin — high oil/gas/trona extraction, arid, fragmented
    '20': 4.5,   # Colorado Plateaus — canyon/desert, extensive extraction, aridity
}
tracts_study['eco_base_score'] = tracts_study['ecoregion_code'].map(ECO_BASE_SCORES)

# ── 4b: HUC-8 water stress ────────────────────────────────────────────────────
ARID_CODES = ['18', '20', '25']
arid_polys = eco_target[eco_target['US_L3CODE'].isin(ARID_CODES)].copy()

HUC8_CACHE = TRACT_DIR / 'huc8_study_area.geojson'
huc8_ok = False

if HUC8_CACHE.exists():
    huc8_gdf = gpd.read_file(HUC8_CACHE)
    huc8_ok = True
    print(f'Loaded cached HUC-8: {len(huc8_gdf)} watersheds')
else:
    bounds = eco_target.total_bounds
    bbox_str = f'{bounds[0]},{bounds[1]},{bounds[2]},{bounds[3]}'
    WBD_URL = 'https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/4/query'
    params = {
        'geometry': bbox_str,
        'geometryType': 'esriGeometryEnvelope',
        'spatialRel': 'esriSpatialRelIntersects',
        'outFields': 'HUC8,NAME',
        'returnGeometry': 'true',
        'outSR': '4326',
        'f': 'geojson',
        'resultRecordCount': '2000',
    }
    print('Fetching HUC-8 watersheds from USGS WBD ...', end=' ', flush=True)
    for attempt in range(2):
        try:
            r = requests.get(WBD_URL, params=params, timeout=90)
            r.raise_for_status()
            huc8_gdf = gpd.read_file(io.StringIO(r.text))
            huc8_gdf = huc8_gdf[huc8_gdf['HUC8'].notna()].copy()
            if len(huc8_gdf) > 0:
                huc8_gdf.to_file(HUC8_CACHE, driver='GeoJSON')
                print(f'OK ({len(huc8_gdf)} watersheds)')
                huc8_ok = True
            else:
                raise ValueError('Empty HUC-8 response')
            break
        except Exception as e:
            if attempt == 0:
                print('retry... ', end='', flush=True)
                time.sleep(8)
            else:
                print(f'FAILED: {e}')
                print('  FALLBACK: water stress assigned from ecoregion lookup.')

if huc8_ok:
    huc8_5070  = huc8_gdf.to_crs('EPSG:5070')
    arid_5070  = arid_polys.to_crs('EPSG:5070').dissolve().reset_index(drop=True)
    huc8_5070['huc8_area_m2'] = huc8_5070.geometry.area

    try:
        huc8_arid = gpd.overlay(
            huc8_5070[['HUC8', 'huc8_area_m2', 'geometry']],
            arid_5070[['geometry']],
            how='intersection',
            keep_geom_type=True,
        )
        huc8_arid['arid_area_m2'] = huc8_arid.geometry.area
        arid_frac = huc8_arid.groupby('HUC8')['arid_area_m2'].sum().reset_index()
        huc8_sc = huc8_5070[['HUC8', 'huc8_area_m2']].merge(arid_frac, on='HUC8', how='left')
        huc8_sc['arid_area_m2'] = huc8_sc['arid_area_m2'].fillna(0)
        huc8_sc['water_stress'] = (
            huc8_sc['arid_area_m2'] / huc8_sc['huc8_area_m2'].replace(0, np.nan)
        ).clip(0, 1)

        huc8_join = huc8_5070[['HUC8', 'geometry']].merge(huc8_sc[['HUC8', 'water_stress']], on='HUC8')
        t_huc = gpd.sjoin(
            tract_cent_5070,
            huc8_join[['HUC8', 'water_stress', 'geometry']],
            how='left', predicate='within',
        ).drop(columns=['index_right'])
        t_huc = t_huc.drop_duplicates(subset=['GEOID'], keep='first')
        tracts_study = tracts_study.merge(t_huc[['GEOID', 'water_stress']], on='GEOID', how='left')
        print(f'  Water stress range: '
              f'{tracts_study["water_stress"].min():.2f} – {tracts_study["water_stress"].max():.2f}')
    except Exception as e:
        print(f'  Water stress overlay failed: {e}. Using fallback.')
        huc8_ok = False

if not huc8_ok:
    fallback_ws = {'21': 0.05, '17': 0.10, '43': 0.40, '25': 0.55,
                   '80': 0.50, '18': 0.80, '20': 0.75}
    tracts_study['water_stress'] = tracts_study['ecoregion_code'].map(fallback_ws)
    print('  Applied fallback water stress from ecoregion lookup.')

# Fill any remaining NaN water_stress with ecoregion median
ws_median = tracts_study['water_stress'].median()
tracts_study['water_stress'] = tracts_study['water_stress'].fillna(ws_median)

# ── 4c: Development pressure proxy ───────────────────────────────────────────
# vacancy_rate (%) from ACS: higher = lower development pressure = better env score
vac_med = tracts_study['vacancy_rate'].median()
tracts_study['development_pressure'] = tracts_study['vacancy_rate'].fillna(vac_med)

print()
print('Environmental indicator summary (study-area tracts):')
for col in ['eco_base_score', 'water_stress', 'development_pressure']:
    s = tracts_study[col].dropna()
    print(f'  {col:<28} mean={s.mean():.2f}  min={s.min():.2f}  max={s.max():.2f}')


Fetching HUC-8 watersheds from USGS WBD ... 

retry... 

FAILED: 'HUC8'
  FALLBACK: water stress assigned from ecoregion lookup.
  Applied fallback water stress from ecoregion lookup.

Environmental indicator summary (study-area tracts):
  eco_base_score               mean=6.37  min=4.50  max=9.00
  water_stress                 mean=0.46  min=0.05  max=0.80
  development_pressure         mean=10.86  min=0.00  max=86.95


### Design Note — Arid Ecosystem Equity

The current E-score structural prior assigns low base scores to arid ecoregions (Wyoming Basin, Colorado Plateaus, Northern Basin and Range) based on extractive land use history and aridity. This is appropriate for the 2024 baseline given observed degradation, but the model architecture must allow these ecoregions to reach E ≥ 8 under restoration scenarios. When NLCD data replaces the structural prior, normalization must be ecoregion-relative (intact sagebrush = high score in Wyoming Basin, intact forest = high score in Southern Rockies). The ceiling is not biome-determined.

## Step 4 — Proxy Decision Documentation

| Indicator | Proxy used | Data source | Limitation | TODO |
|-----------|-----------|-------------|------------|------|
| Land cover / ecological condition | Ecoregion base score (structural prior) | EPA Level III Ecoregion Descriptions (2011) | Constant within ecoregion; ignores intra-ecoregion variation | TODO: replace with NLCD 2021 land cover % composition per tract |
| Water stress | HUC-8 arid ecoregion fraction | USGS WBD REST API + EPA L3 ecoregion polygons | Aggregated at watershed scale; no temporal drought signal | TODO: replace with USDA NRCS monthly water balance or NOAA U.S. Drought Monitor |
| Development pressure | Housing vacancy rate (inverted) | ACS B25002 (2022) | Vacancy driven by many factors (seasonal homes, economic decline) beyond development pressure | TODO: replace with NLCD impervious surface change 2011–2021 per tract |
| University presence | HIFLD Colleges and Universities layer | HIFLD FeatureServer (2024) | Includes all post-secondary institutions, not just degree-granting universities | TODO: validate against NCES IPEDS 2022 postsecondary institution database |

All proxy indicators are flagged in `network_metadata.json → ees_baseline → todo`.


## Step 5 — Institutional Anchors

Hospitals and universities from HIFLD open data. Government employment share
from ACS C24060 (already fetched in Step 2).

- **Hospitals:** HIFLD Hospitals FeatureServer. If fails → `hospital_count = NaN` (flagged).
- **Universities:** HIFLD Colleges and Universities FeatureServer. If fails → `has_university = 0`.
- **Government employment:** `(local + state + federal govt workers) / total workers × 100` from ACS C24060.


In [7]:
STATES_FILTER = "STATE IN ('WY','CO','MT','UT','NM','ID','NV','NE','SD')"


def fetch_hifld(url, where_clause, out_fields, retries=2):
    params = {
        'where': where_clause,
        'outFields': out_fields,
        'returnGeometry': 'true',
        'outSR': '4326',
        'f': 'geojson',
        'resultRecordCount': '5000',
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            gdf = gpd.read_file(io.StringIO(r.text))
            return gdf[gdf.geometry.notna()].copy()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(5)
            else:
                return None


# ── Hospitals ─────────────────────────────────────────────────────────────────
HOSP_URL = ('https://services1.arcgis.com/Hp6G80Pky0om7QvQ/arcgis/rest/'
            'services/Hospitals/FeatureServer/0/query')
print('Fetching hospitals from HIFLD ...', end=' ', flush=True)
hosp_gdf = fetch_hifld(HOSP_URL, STATES_FILTER, 'OBJECTID,NAME,STATE')
if hosp_gdf is not None and len(hosp_gdf) > 0:
    print(f'OK ({len(hosp_gdf)} hospitals)')
    hosp_ok = True
else:
    print('FAILED — hospital_count will be NaN (flag for manual update)')
    hosp_gdf, hosp_ok = None, False

# ── Universities ──────────────────────────────────────────────────────────────
UNIV_URL = ('https://services1.arcgis.com/Hp6G80Pky0om7QvQ/arcgis/rest/'
            'services/Colleges_Universities/FeatureServer/0/query')
print('Fetching universities from HIFLD ...', end=' ', flush=True)
univ_gdf = fetch_hifld(UNIV_URL, STATES_FILTER, 'OBJECTID,NAME,STATE')
if univ_gdf is not None and len(univ_gdf) > 0:
    print(f'OK ({len(univ_gdf)} institutions)')
    univ_ok = True
else:
    print('FAILED — has_university will be 0 (flag for manual update)')
    univ_gdf, univ_ok = None, False

# ── Spatial join to tracts ────────────────────────────────────────────────────
tracts_5070_poly = tracts_study[['GEOID', 'geometry']].to_crs('EPSG:5070')

if hosp_ok:
    hosp_in = gpd.sjoin(
        hosp_gdf.to_crs('EPSG:5070')[['geometry']],
        tracts_5070_poly,
        how='inner', predicate='within',
    )
    hosp_ct = hosp_in.groupby('GEOID').size().reset_index(name='hospital_count')
    tracts_study = tracts_study.merge(hosp_ct, on='GEOID', how='left')
    tracts_study['hospital_count'] = tracts_study['hospital_count'].fillna(0).astype(int)
else:
    tracts_study['hospital_count'] = np.nan

if univ_ok:
    univ_in = gpd.sjoin(
        univ_gdf.to_crs('EPSG:5070')[['geometry']],
        tracts_5070_poly,
        how='inner', predicate='within',
    )
    tracts_study['has_university'] = tracts_study['GEOID'].isin(univ_in['GEOID'].unique()).astype(int)
else:
    tracts_study['has_university'] = 0

print()
if hosp_ok:
    n_hosp = int(tracts_study['hospital_count'].sum())
    n_tract_hosp = (tracts_study['hospital_count'] > 0).sum()
    print(f'  Hospital count: {n_hosp} total | {n_tract_hosp} tracts with ≥1 hospital')
if univ_ok:
    print(f'  University presence: {tracts_study["has_university"].sum()} tracts with ≥1 institution')
gs_med = tracts_study['govt_employment_share'].median()
print(f'  Govt employment share: median = {gs_med:.1f}%')


Fetching hospitals from HIFLD ... 

FAILED — hospital_count will be NaN (flag for manual update)
Fetching universities from HIFLD ... 

FAILED — has_university will be 0 (flag for manual update)



  Govt employment share: median = 82.9%


## Step 6 — Scoring and Aggregation

Each indicator normalized to 0–10 using the **Mountain West study-area distribution**
(min-max across all study-area tracts). Direction: higher is always better after inversion.

**Environmental (E):** eco_base_score · water_stress (inverted) · development_pressure
**Economic (Ec):** income · employment_rate · poverty (inverted) · vacancy (inverted) · gen_cap_50km · dist_to_bus (inverted)
**Social (S):** education · health_insurance · housing_age · internet · hospitals · university · govt_employment

Ecoregion-level scores: population-weighted average across tract scores.


In [8]:
def minmax_norm(series, lo=0.0, hi=10.0, invert=False):
    s = series.astype(float)
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(5.0, index=s.index)
    n = (s - mn) / (mx - mn) * (hi - lo) + lo
    return (hi + lo - n) if invert else n


scoring_df = tracts_study[tracts_study['n_missing'] <= 3].copy().reset_index(drop=True)
print(f'Tracts entering scoring: {len(scoring_df):,}')

# ── Environmental ─────────────────────────────────────────────────────────────
scoring_df['E_eco_base'] = minmax_norm(scoring_df['eco_base_score'])
scoring_df['E_water']    = minmax_norm(scoring_df['water_stress'], invert=True)
scoring_df['E_dev']      = minmax_norm(
    scoring_df['development_pressure'].fillna(scoring_df['development_pressure'].median())
)
scoring_df['E_score'] = scoring_df[['E_eco_base', 'E_water', 'E_dev']].mean(axis=1)

# ── Economic ──────────────────────────────────────────────────────────────────
scoring_df['Ec_income']  = minmax_norm(scoring_df['median_hh_income'])
scoring_df['Ec_employ']  = minmax_norm(scoring_df['employment_rate'])
scoring_df['Ec_poverty'] = minmax_norm(scoring_df['poverty_rate'], invert=True)
scoring_df['Ec_vacancy'] = minmax_norm(scoring_df['vacancy_rate'],  invert=True)
scoring_df['Ec_gencap']  = minmax_norm(scoring_df['gen_cap_50km_mw'])
scoring_df['Ec_tx']      = minmax_norm(scoring_df['dist_to_bus_km'], invert=True)
scoring_df['Ec_score'] = scoring_df[
    ['Ec_income', 'Ec_employ', 'Ec_poverty', 'Ec_vacancy', 'Ec_gencap', 'Ec_tx']
].mean(axis=1)

# ── Social ────────────────────────────────────────────────────────────────────
scoring_df['S_edu']      = minmax_norm(scoring_df['pct_bachelor_plus'])
scoring_df['S_health']   = minmax_norm(scoring_df['pct_health_insurance'])
scoring_df['S_housing']  = minmax_norm(scoring_df['median_year_built'])
scoring_df['S_internet'] = minmax_norm(scoring_df['pct_internet'])
scoring_df['S_hospital'] = (
    minmax_norm(scoring_df['hospital_count'].fillna(0))
    if scoring_df['hospital_count'].notna().sum() > 10
    else pd.Series(5.0, index=scoring_df.index)
)
scoring_df['S_univ'] = scoring_df['has_university'].astype(float) * 10.0
scoring_df['S_govt'] = minmax_norm(
    scoring_df['govt_employment_share'].fillna(
        scoring_df['govt_employment_share'].median()
    )
)
scoring_df['S_score'] = scoring_df[
    ['S_edu', 'S_health', 'S_housing', 'S_internet', 'S_hospital', 'S_univ', 'S_govt']
].mean(axis=1)

scoring_df['composite_score'] = scoring_df[['E_score', 'Ec_score', 'S_score']].mean(axis=1)

# ── Population-weighted ecoregion aggregation ─────────────────────────────────
scoring_df['_pop_w'] = scoring_df['population'].fillna(0).clip(lower=0)

def _wavg(grp):
    w = grp['_pop_w']
    tw = w.sum()
    out = {}
    for c in ['E_score', 'Ec_score', 'S_score', 'composite_score']:
        out[c] = (grp[c] * w).sum() / tw if tw > 0 else grp[c].mean()
    return pd.Series(out)

eco_scores = (
    scoring_df.groupby(['ecoregion_code', 'ecoregion_name'])
    .apply(_wavg)
    .reset_index()
)
eco_counts = (
    scoring_df.groupby('ecoregion_code')
    .agg(tract_count=('GEOID', 'count'), population_total=('population', 'sum'))
    .reset_index()
)
eco_summary = (
    eco_scores.merge(eco_counts, on='ecoregion_code')
    .sort_values('composite_score', ascending=False)
    .reset_index(drop=True)
)

# ── Save outputs ──────────────────────────────────────────────────────────────
SCORE_COLS = [
    'GEOID', 'ecoregion_code', 'ecoregion_name', 'state_abbr',
    'population', 'median_hh_income', 'poverty_rate', 'vacancy_rate',
    'employment_rate', 'pct_bachelor_plus', 'pct_health_insurance',
    'median_year_built', 'pct_internet', 'govt_employment_share',
    'gen_cap_50km_mw', 'dist_to_bus_km', 'eco_base_score',
    'water_stress', 'development_pressure', 'hospital_count', 'has_university',
    'n_missing',
    'E_eco_base', 'E_water', 'E_dev',
    'Ec_income', 'Ec_employ', 'Ec_poverty', 'Ec_vacancy', 'Ec_gencap', 'Ec_tx',
    'S_edu', 'S_health', 'S_housing', 'S_internet', 'S_hospital', 'S_univ', 'S_govt',
    'E_score', 'Ec_score', 'S_score', 'composite_score',
]
save_cols = [c for c in SCORE_COLS if c in scoring_df.columns]
scoring_df[save_cols].to_parquet(ROOT / 'data/processed/mw_tract_ees_scores.parquet', index=False)

eco_summary.to_csv(ROOT / 'data/processed/mw_ecoregion_ees_summary.csv', index=False)

print(f'\nSaved: mw_tract_ees_scores.parquet  ({len(scoring_df):,} tracts, {len(save_cols)} columns)')
print(f'Saved: mw_ecoregion_ees_summary.csv ({len(eco_summary)} ecoregions)')
print()
disp = ['ecoregion_code', 'ecoregion_name', 'E_score', 'Ec_score', 'S_score',
        'composite_score', 'tract_count']
print('Ecoregion EES Summary (population-weighted, ranked by composite score):')
print(eco_summary[disp].to_string(index=False, float_format='%.2f'))


Tracts entering scoring: 1,676

Saved: mw_tract_ees_scores.parquet  (1,676 tracts, 42 columns)
Saved: mw_ecoregion_ees_summary.csv (7 ecoregions)

Ecoregion EES Summary (population-weighted, ranked by composite score):
ecoregion_code            ecoregion_name  E_score  Ec_score  S_score  composite_score  tract_count
            21          Southern Rockies     7.52      6.28     5.39             6.40          203
            17            Middle Rockies     6.56      5.84     5.21             5.87          156
            25               High Plains     2.40      7.56     5.43             5.13          937
            43 Northwestern Great Plains     2.96      5.82     4.85             4.55          175
            80  Northern Basin and Range     2.06      5.99     4.96             4.34           19
            20         Colorado Plateaus     0.69      6.04     4.96             3.89          112
            18             Wyoming Basin     0.60      5.78     4.85             3.74   

## Step 7 — Summary Table, Folium Choropleth Map, and Metadata Update


In [9]:
import math

# ── 7a: Within-ecoregion range table ─────────────────────────────────────────
print('=' * 70)
print('ECOREGION EES SUMMARY  (ranked by composite resilience score)')
print('=' * 70)
disp = ['ecoregion_code', 'ecoregion_name', 'E_score', 'Ec_score', 'S_score', 'composite_score', 'tract_count']
print(eco_summary[disp].to_string(index=False, float_format='%.2f'))
print()
print('Within-ecoregion range (highest / lowest composite scoring tracts):')
for _, row in eco_summary.iterrows():
    code = row['ecoregion_code']
    sub  = scoring_df[scoring_df['ecoregion_code'] == code].sort_values('composite_score', ascending=False)
    if len(sub) == 0:
        continue
    hi, lo = sub.iloc[0], sub.iloc[-1]
    print(f'  {row["ecoregion_name"]}')
    print(f'    High: GEOID {hi["GEOID"]}  composite={hi["composite_score"]:.2f}  '
          f'(E={hi["E_score"]:.2f} Ec={hi["Ec_score"]:.2f} S={hi["S_score"]:.2f})')
    print(f'    Low:  GEOID {lo["GEOID"]}  composite={lo["composite_score"]:.2f}  '
          f'(E={lo["E_score"]:.2f} Ec={lo["Ec_score"]:.2f} S={lo["S_score"]:.2f})')
print()

# ── 7b: Folium choropleth ────────────────────────────────────────────────────
print('Building Folium choropleth map ...', end=' ', flush=True)

map_gdf = tracts_study[['GEOID', 'geometry']].merge(
    scoring_df[['GEOID', 'ecoregion_code', 'composite_score', 'E_score', 'Ec_score', 'S_score']],
    on='GEOID', how='inner',
).copy()
# Simplify geometries for web rendering (~1 km tolerance at mid-latitude)
map_gdf['geometry'] = map_gdf.geometry.simplify(0.01, preserve_topology=True)

m = folium.Map(location=[42.5, -108.0], zoom_start=5, tiles='CartoDB positron')

score_series = scoring_df.set_index('GEOID')['composite_score']
folium.Choropleth(
    geo_data=map_gdf[['GEOID', 'geometry']].to_json(),
    data=score_series,
    key_on='feature.properties.GEOID',
    fill_color='RdYlGn',
    fill_opacity=0.70,
    line_opacity=0.10,
    line_color='white',
    legend_name='EES Composite Resilience Score (0–10)',
    nan_fill_color='lightgrey',
    name='EES Composite Score',
).add_to(m)

# Ecoregion outlines
eco_dissolved = (
    eco_target.dissolve(by='US_L3CODE').reset_index()[['US_L3CODE', 'US_L3NAME', 'geometry']]
)
folium.GeoJson(
    eco_dissolved,
    name='Ecoregion outlines',
    style_function=lambda f: {
        'fillColor': 'none',
        'color': 'white',
        'weight': 2.5,
        'dashArray': '6 3',
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['US_L3CODE', 'US_L3NAME'], aliases=['Code', 'Ecoregion']
    ),
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

MAP_OUT = ROOT / 'data/processed/figures/ees_baseline_map.html'
m.save(str(MAP_OUT))
print(f'OK → {MAP_OUT.name}  ({MAP_OUT.stat().st_size // 1024:,} KB)')

# ── 7c: Update network_metadata.json ─────────────────────────────────────────
meta['ees_baseline'] = {
    'tract_count': int(len(scoring_df)),
    'ecoregion_scores': {
        row['ecoregion_code']: {
            'name':      row['ecoregion_name'],
            'E_score':   round(float(row['E_score']),   2),
            'Ec_score':  round(float(row['Ec_score']),  2),
            'S_score':   round(float(row['S_score']),   2),
            'composite': round(float(row['composite_score']), 2),
        }
        for _, row in eco_summary.iterrows()
    },
    'data_vintage': 'ACS 2022 5-year, TIGER 2020, EIA power plants 2024',
    'proxy_indicators': [
        'eco_base_score: EPA L3 structural prior; TODO replace with NLCD land cover',
        'water_stress: HUC-8 arid fraction; TODO replace with USDA NRCS water balance',
        'development_pressure: vacancy_rate proxy; TODO replace with NLCD impervious change',
        'has_university: HIFLD Colleges layer; TODO validate against NCES IPEDS 2022',
    ],
    'scoring_notes': (
        'Indicators normalized 0-10 using Mountain West distribution. '
        'Population-weighted ecoregion aggregation. '
        'Government employment from ACS C24060 (class of worker rows 5+6+7), '
        'not C24050 (industry sector, row 3 = Agriculture). '
        'Tracts with >3 missing indicators excluded.'
    ),
    'todo': [
        'Replace eco_base_score with NLCD 2021 land cover % composition per tract',
        'Replace water_stress with USDA NRCS or NOAA Drought Monitor raster',
        'Replace development_pressure with NLCD impervious surface change 2011-2021',
        'Validate university flag against NCES IPEDS 2022 database',
    ],
    'timestamp': pd.Timestamp.now().isoformat(),
}

with open(ROOT / 'data/processed/network_metadata.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print('Updated network_metadata.json  (ees_baseline block added)')


ECOREGION EES SUMMARY  (ranked by composite resilience score)
ecoregion_code            ecoregion_name  E_score  Ec_score  S_score  composite_score  tract_count
            21          Southern Rockies     7.52      6.28     5.39             6.40          203
            17            Middle Rockies     6.56      5.84     5.21             5.87          156
            25               High Plains     2.40      7.56     5.43             5.13          937
            43 Northwestern Great Plains     2.96      5.82     4.85             4.55          175
            80  Northern Basin and Range     2.06      5.99     4.96             4.34           19
            20         Colorado Plateaus     0.69      6.04     4.96             3.89          112
            18             Wyoming Basin     0.60      5.78     4.85             3.74           74

Within-ecoregion range (highest / lowest composite scoring tracts):
  Southern Rockies
    High: GEOID 08117000403  composite=7.23  (E=9.35 Ec=6.

OK → ees_baseline_map.html  (6,099 KB)
Updated network_metadata.json  (ees_baseline block added)


In [10]:
# ── Confirmation: all outputs exist ──────────────────────────────────────────
outputs = {
    'mw_tract_ees_scores.parquet':  ROOT / 'data/processed/mw_tract_ees_scores.parquet',
    'mw_ecoregion_ees_summary.csv': ROOT / 'data/processed/mw_ecoregion_ees_summary.csv',
    'ees_baseline_map.html':        ROOT / 'data/processed/figures/ees_baseline_map.html',
    'network_metadata.json':        ROOT / 'data/processed/network_metadata.json',
}

print('=' * 65)
print('CONFIRMATION — Output files')
print('=' * 65)
all_ok = True
for name, path in outputs.items():
    if path.exists():
        kb = path.stat().st_size / 1024
        print(f'  OK     {name:<40} {kb:>8.1f} KB')
    else:
        print(f'  MISSING  {name}')
        all_ok = False
print('=' * 65)
if all_ok:
    print('  ALL OUTPUTS SAVED  ✓')
else:
    print('  SOME OUTPUTS MISSING  ✗')
print('=' * 65)

# Spot-checks
ees_df   = pd.read_parquet(ROOT / 'data/processed/mw_tract_ees_scores.parquet')
eco_csv  = pd.read_csv(ROOT / 'data/processed/mw_ecoregion_ees_summary.csv')
print(f'\nSpot-check mw_tract_ees_scores: {ees_df.shape}')
print(f'  composite_score range: {ees_df["composite_score"].min():.2f} – {ees_df["composite_score"].max():.2f}')
print(f'  ecoregion distribution:')
print(ees_df.groupby('ecoregion_code')['GEOID'].count().to_string())
print(f'\nSpot-check mw_ecoregion_ees_summary ({len(eco_csv)} ecoregions):')
print(eco_csv[['ecoregion_code', 'ecoregion_name', 'E_score', 'Ec_score', 'S_score', 'composite_score']]
      .to_string(index=False, float_format='%.2f'))


CONFIRMATION — Output files
  OK     mw_tract_ees_scores.parquet                 324.4 KB
  OK     mw_ecoregion_ees_summary.csv                  0.8 KB
  OK     ees_baseline_map.html                      6099.3 KB
  OK     network_metadata.json                        10.9 KB
  ALL OUTPUTS SAVED  ✓

Spot-check mw_tract_ees_scores: (1676, 42)
  composite_score range: 2.23 – 7.23
  ecoregion distribution:
ecoregion_code
17    156
18     74
20    112
21    203
25    937
43    175
80     19

Spot-check mw_ecoregion_ees_summary (7 ecoregions):
 ecoregion_code            ecoregion_name  E_score  Ec_score  S_score  composite_score
             21          Southern Rockies     7.52      6.28     5.39             6.40
             17            Middle Rockies     6.56      5.84     5.21             5.87
             25               High Plains     2.40      7.56     5.43             5.13
             43 Northwestern Great Plains     2.96      5.82     4.85             4.55
             80  Nort